In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
dataset = pd.read_csv('train-2.csv')

print(dataset.shape)
print(dataset.isna().sum())
dataset.head()

dict_for_coding = {'male' : 1, 'female' : 0}

dataset['Sex_coding'] = [dict_for_coding[i] for i in dataset['Sex']]
features_for_predict = ['Pclass', 'Sex_coding','Age','Fare',]
X,y = dataset[features_for_predict], dataset['Survived']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
age_median = X_train['Age'].median()

X_train = X_train.copy()
X_test = X_test.copy()


X_train['Age'] = X_train['Age'].fillna(age_median)
X_test['Age'] = X_test['Age'].fillna(age_median)


(891, 12)
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [2]:
print(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()),X_train,y_train,cv = 5, scoring='accuracy'))

[0.7972028  0.82517483 0.75352113 0.74647887 0.83098592]


In [3]:
C = [0.001, 0.01, 0.1, 1, 10, 100]
for i in C:
    print(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C = i)),X_train,y_train,cv = 5, scoring='accuracy').mean())


0.6320397911947208
0.7766374470599823
0.7920811582783414
0.7906727075741161
0.7906727075741161
0.7906727075741161


2 [20.7972028 0.82517483 0.75352113 0.74647887 0.83098592] среднее из полученых значений(0.79067) находится очень близка с 0.793 это говорит о стабильном результате работы модели
3 при высоких значениях регуляризации модель имеет низкий результат около 0.63 потому что штраф настолько велик что модель не может использовать признаки  начтная со значения 1 и до 100 результаты модели не изменяются, лучшее значение С будет 1 10 или 100 их значения почти не отличаются так как штраф очень слабый,переобучения в данных почти что нет и регуляризация мало что меняет, при малом С результаты заметно ухудщаются
4 при С = 1 получили результаты [0.7972028  0.82517483 0.75352113 0.74647887 0.83098592]

    при С от 0.001 ,0.01,0.1,1,10,100
получились следующие значения
[0.62937063 0.62237762 0.64788732 0.64084507 0.61971831]
[0.76223776 0.82517483 0.76760563 0.75352113 0.77464789]
[0.7972028  0.82517483 0.76760563 0.74647887 0.82394366]
[0.7972028  0.82517483 0.75352113 0.74647887 0.83098592]
[0.7972028  0.82517483 0.75352113 0.74647887 0.83098592]
[0.7972028  0.82517483 0.75352113 0.74647887 0.83098592]

## Вывод: кросс-валидация и параметр C (Titanic, LogisticRegression)

**2. Кросс-валидация (5 фолдов, C = 1).** Оценки по фолдам: [0.7972028, 0.82517483, 0.75352113, 0.74647887, 0.83098592]. Среднее из полученных значений (0.79067) очень близко к 0.793 (результат на отложенной части), это говорит о стабильном результате работы модели. При этом отдельные фолды колеблются от 0.75 до 0.83 (std ≈ 0.035), поэтому на одном разбиении результат мог быть и хуже, и лучше.

**3. Влияние C.** Параметр C это 1/λ: малый C означает сильную регуляризацию.

| C | Среднее CV-accuracy |
|---|---|
| 0.001 | 0.632 |
| 0.01 | 0.777 |
| 0.1 | 0.792 |
| 1 | 0.791 |
| 10 | 0.791 |
| 100 | 0.791 |

- При высоких значениях регуляризации (C = 0.001) модель имеет низкий результат, около 0.63, потому что штраф настолько велик, что модель не может использовать признаки (недообучение). При C = 0.01 результаты тоже заметно хуже.
- Начиная со значения 0.1 и до 100 результаты модели почти не меняются, лучшее среднее у C = 0.1 (0.792), но разница с C = 1, 10 и 100 (0.791) в пределах шума. Значения почти не отличаются, так как штраф очень слабый, переобучения в данных почти нет и регуляризация мало что меняет.

**4. Вывод.** Для этих данных (4 признака, 712 строк) модель не переобучается, и подбор C даёт выигрыш только в сторону отказа от слишком сильной регуляризации. Подходит любое C от 0.1 до 100.